In [30]:
import torch
from torch.utils.data import Dataset, DataLoader
import json
import os
import cv2
import numpy as np
from glob import glob
from natsort import natsorted

from collections import deque

In [52]:
# number of frames to group
f = 3
# number of head positions to keep
k = 15
fps = 30

vd_paths = natsorted(
    glob(
        "/home/alien/Documents/access/processed_dataset/eht_dataset_salvideos/**/**/*.mp4"
    )
)


for vd_path in vd_paths[0:1]:
    frames = []
    head_positions = []

    print(vd_path)
    json_meta_path = os.path.splitext(vd_path)[0] + ".json"
    print(json_meta_path)
    if os.path.exists(vd_path) and os.path.exists(json_meta_path):
        with open(json_meta_path, "r") as file:
            json_data = json.load(file)
        print(len(json_data))

        # Initialize a deque to keep the last frames
        last_frames = deque(maxlen=f)
        cap = cv2.VideoCapture(vd_path)
        num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(num_frames)

        frame_interval = 30 // 6
        frame_idx = 0
        # frame_idx = k + f - 1
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            last_frames.append(frame)
            frame_idx += 1
            for _ in range(k):
                ret, frame = cap.read()
                if not ret:
                    break
                if frame_idx % frame_interval == 0:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    last_frames.append(frame)
                frame_idx += 1
            else:
                if len(last_frames) < f:
                    continue
                head_positions.append(
                    [
                        [data["frame_number"], data["giw_long"], data["giw_lat"]]
                        for data in json_data[frame_idx - k : frame_idx]
                    ]
                )
                frames.append(list(last_frames))
                continue
            break
        else:
            raise ValueError("Failed to read frame from video:", vd_path)

        cap.release()

/home/alien/Documents/access/processed_dataset/eht_dataset_salvideos/Video_1/User_1_Task_1/Video_1_User_1_Task_1_1.mp4
/home/alien/Documents/access/processed_dataset/eht_dataset_salvideos/Video_1/User_1_Task_1/Video_1_User_1_Task_1_1.json
150
150


In [57]:
len(head_positions[0])

15

In [58]:
np.array(frames).shape

(9, 3, 288, 384)

In [107]:
# number of frames to group
f = 2
# number of head positions to keep
k = 10
fps = 30

vd_paths = natsorted(glob("../data/results/EHT_dataset_salvideos/**/**/*.mp4"))

frames = []
head_positions = []
for vd_path in vd_paths[0:1]:
    frame_idx = k + f - 1
    print(vd_path)
    json_meta_path = os.path.splitext(vd_path)[0] + ".json"
    print(json_meta_path)
    if os.path.exists(vd_path) and os.path.exists(json_meta_path):
        with open(json_meta_path, "r") as file:
            json_data = json.load(file)
        print(len(json_data))

        # frames = []
        cap = cv2.VideoCapture(vd_path)
        num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(k + f - 1))
        while cap.isOpened():
            b_frames = []
            for _ in range(f):
                ret, frame = cap.read()
                if not ret:
                    break
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                b_frames.append(frame)
                frame_idx += 1
            else:
                head_positions.append(
                    [
                        [data["frame_number"], data["giw_long"], data["giw_lat"]]
                        for data in json_data[frame_idx - k : frame_idx]
                    ]
                )
                frames.append(b_frames)
                continue
            break
        else:
            raise ValueError("Failed to read frame from video:", vd_path)

        cap.release()

In [ ]:
import os
import json
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from natsort import natsorted
from glob import glob


class CustomDataset(Dataset):
    def __init__(self, vd_paths, k=10, f=2):
        self.vd_paths = vd_paths
        self.k = k
        self.f = f

    def __len__(self):
        return len(self.vd_paths)

    def __getitem__(self, idx):
        vd_path = self.vd_paths[idx]
        frames = []
        head_positions = []

        frame_idx = self.k - 1
        json_meta_path = os.path.splitext(vd_path)[0] + ".json"

        if os.path.exists(vd_path) and os.path.exists(json_meta_path):
            with open(json_meta_path, "r") as file:
                json_data = json.load(file)

            cap = cv2.VideoCapture(vd_path)
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
            while cap.isOpened():
                b_frames = []
                for _ in range(self.f):
                    ret, frame = cap.read()
                    if not ret:
                        break
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    b_frames.append(frame)
                    frame_idx += 1
                else:
                    head_positions.append(
                        [
                            [
                                data["frame_number"],
                                data["video_id"],
                                data["user_id"],
                                data["task_id"],
                                data["giw_long"],
                                data["giw_lat"],
                            ]
                            for data in json_data[
                                frame_idx - self.k - 1 : frame_idx - 1
                            ]
                        ]
                    )
                    frames.append(b_frames)
                    continue
                break
            else:
                raise ValueError("Failed to read frame from video:", vd_path)

            cap.release()

        return torch.tensor(frames), torch.tensor(head_positions)


def custom_collate(batch):
    # Filter out None samples
    batch = [sample for sample in batch if sample is not None]
    if len(batch) == 0:
        return None  # Return None if all samples are None

    frames_batch, head_positions_batch = zip(*batch)

    # Flatten the first dimension
    frames_flat = torch.cat(frames_batch, dim=0)
    head_positions_flat = torch.cat(head_positions_batch, dim=0)

    # # Shuffle the flattened data
    # indices = torch.randperm(frames_flat.size(0))
    # frames_flat = frames_flat[indices]
    # head_positions_flat = head_positions_flat[indices]

    # Define your batch sizes
    batch_sizes = [32]  # Example batch sizes

    # Create batches of different sizes
    batches = []
    start = 0
    for batch_size in batch_sizes:
        end = start + batch_size
        batches.append((frames_flat[start:end], head_positions_flat[start:end]))
        start = end

    return batches


# Define your vd_paths
vd_paths = natsorted(glob("../data/results/EHT_dataset_salvideos/**/**/*.mp4"))

# Create an instance of your custom dataset
dataset = CustomDataset(vd_paths)

# Create a PyTorch DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    pin_memory=True,
    num_workers=4,
    collate_fn=custom_collate,
    drop_last=True,
)


batch_num = 0
for batch in dataloader:
    # if batch is None:
    #     continue
    frames, head_positions = batch[0]
    batch_num += 1
    print(f"Batch {batch_num}")
    print(np.array(frames).shape, np.array(head_positions).shape)

In [106]:
import sys
from natsort import natsorted
from glob import glob
sys.path.append('..')
from dataloader import get_dataloader
import numpy as np


%load_ext autoreload
%autoreload 2


# Define your vd_paths
vd_paths = natsorted(glob('../../data/results/EHT_dataset_salvideos/**/**/*.mp4'))

dataloader = get_dataloader(vd_paths, batch_size=2, num_workers=0, shuffle=False)
batch_num=0
for batch in dataloader:
    print('here')
    if batch is None:
        continue
    for b in batch:
        print(b[0].shape,np.array(b[1]).astype(int))
    # frames, head_positions = batch
    # batch_num+=1
    # print(f"Batch {batch_num}")
    # print(np.array(frames).shape, np.array(head_positions).shape)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from torchtsmixer import TSMixer

from torchinfo import summary

import sys
from natsort import natsorted
from glob import glob
sys.path.append('..')
from dataloader import get_dataloader
import numpy as np

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm

class Conv_tsmixer(nn.Module):
    def __init__(
        self,
        num_time_features,
        conv_output_embeddings,
        sequence_length,
        prediction_length,
        prediction_features,
        image_kernel_size=5,  # Default kernel size for convolutions
        image_padding=0,
    ):
        super(Conv_tsmixer, self).__init__()

        # Convolutional layers for each image (separate branches)
        self.conv1 = nn.Conv2d(
            1, 16, kernel_size=image_kernel_size, padding=image_padding
        )
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(
            16,
            8,
            kernel_size=image_kernel_size,
            padding=image_padding,
        )
        self.bn2 = nn.BatchNorm2d(8)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        self.fc1 = nn.Linear(8 * 3 * 3, conv_output_embeddings)

        # Input channel size adjusted to consider individual image features
        self.tsmixer = TSMixer(
            sequence_length=sequence_length,
            prediction_length=prediction_length,
            input_channels=conv_output_embeddings
            + num_time_features,  # Adjusted input channels
            output_channels=prediction_features,
        )

    def forward(self, image, time_series):
        # Image processing for each image (separate branches)
        batch_size = image.size(0)
        num_images = image.size(1)
        num_time_steps = time_series.size(1)

        image_features = []
        for i in range(num_images):  # Process each image in the batch
            img = image[:, i, :, :]  # Select the ith image
            img = img.unsqueeze(1)  # Add channel dimension
            x = F.interpolate(img, size=(25, 25))  # Resize
            x = self.pool(F.relu(self.bn1(self.conv1(x))))
            x = self.pool(F.relu(self.bn2(self.conv2(x))))
            x = x.view(batch_size, -1)  # Flatten
            x = F.relu(self.fc1(x))
            # Expand features to match time series length and concatenate with time series data
            image_features.append(x.unsqueeze(1))
        img_features = torch.cat(image_features, dim=1)
        # print(img_features.shape)

        linspace_imgs_to_time_series = torch.linspace(
            0, num_time_steps, num_images + 1, dtype=torch.int
        )
        features = torch.zeros(8, 15, 10, device="cuda")

        # Copy the slices from A to the appropriate ranges in B
        for i in range(num_images):
            start_idx = linspace_imgs_to_time_series[i]
            end_idx = linspace_imgs_to_time_series[i + 1]
            features[:, start_idx:end_idx, :] = (
                img_features[:, i, :].unsqueeze(1).repeat(1, end_idx - start_idx, 1)
            )

        # print(features.shape)  # Should print torch.Size([8, 15, 10])
        # Concatenate time series data with features from all images
        combined_features = torch.cat((time_series, features), dim=2)
        # print(combined_features.shape)
        # # Pass the concatenated features through the TSMixer
        output = self.tsmixer(combined_features)

        return output



# Example usage
if __name__ == "__main__":
    batch_size = 8
    num_workers = 4
    num_time_features = 4
    conv_output_embeddings = 10
    sequence_length = 15
    prediction_length = 1
    prediction_features = 4
    num_epochs = 10
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # Create dummy inputs
    # image_input = torch.randn(batch_size, 3, 1, 224, 224)
    # time_series_input = torch.randn(batch_size, sequence_length, num_time_features)

    # Initialize the model
    model = Conv_tsmixer(
        num_time_features,
        conv_output_embeddings,
        sequence_length,
        prediction_length,
        prediction_features,
    ).to(device)

    # Loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    # Define your vd_paths
    vd_paths = natsorted(glob(
            "/home/alien/Documents/access/processed_dataset/eht_dataset_salvideos/**/**/*.mp4"
        ))
    
    # Split the video paths into training and validation sets
    train_vd_paths = vd_paths[:int(0.8 * len(vd_paths))]
    val_vd_paths = vd_paths[int(0.8 * len(vd_paths)):]

    train_dataloader = get_dataloader(
        train_vd_paths, batch_size=batch_size, num_workers=num_workers
    )
    val_dataloader = get_dataloader(
        val_vd_paths, batch_size=batch_size, num_workers=num_workers
    )

    # TensorBoard writer
    writer = SummaryWriter()

    for epoch in tqdm(range(num_epochs)):
        model.train()
        epoch_loss = 0.0
        for batches in train_dataloader:
            for b in batches:
                images, timeseries, predictions = b
                images, timeseries, predictions = images.to(device), timeseries.to(device), predictions.to(device)
                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                outputs = model(images, timeseries)
                loss = criterion(outputs, predictions)

                # Backward pass and optimize
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
        
        avg_epoch_loss = epoch_loss / len(train_dataloader)
        print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {avg_epoch_loss:.4f}')
        
        # Log the training loss to TensorBoard
        writer.add_scalar('Training Loss', avg_epoch_loss, epoch)
        
        # Validation step
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batches in val_dataloader:
                for b in batches:
                    images, timeseries, predictions = b
                    images, timeseries, predictions = images.to(device), timeseries.to(device), predictions.to(device)
                    
                    # Forward pass
                    outputs = model(images, timeseries)
                    loss = criterion(outputs, predictions)
                    
                    val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_dataloader)
        print(f'Epoch [{epoch+1}/{num_epochs}], Validation Loss: {avg_val_loss:.4f}')
        
        # Log the validation loss to TensorBoard
        writer.add_scalar('Validation Loss', avg_val_loss, epoch)

    # Close the TensorBoard writer
    writer.close()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


  0%|          | 0/10 [06:08<?, ?it/s]


KeyboardInterrupt: 

In [11]:
b[0].shape

torch.Size([8, 3, 288, 384])

In [6]:
summary(model, input_size=[(8, 3, 224, 224), (8, 15, 4)])

Layer (type:depth-idx)                             Output Shape              Param #
Conv_tsmixer                                       [8, 4, 1]                 --
├─Conv2d: 1-1                                      [8, 16, 21, 21]           416
├─BatchNorm2d: 1-2                                 [8, 16, 21, 21]           32
├─MaxPool2d: 1-3                                   [8, 16, 10, 10]           --
├─Conv2d: 1-4                                      [8, 8, 6, 6]              3,208
├─BatchNorm2d: 1-5                                 [8, 8, 6, 6]              16
├─MaxPool2d: 1-6                                   [8, 8, 3, 3]              --
├─Linear: 1-7                                      [8, 10]                   730
├─Conv2d: 1-8                                      [8, 16, 21, 21]           (recursive)
├─BatchNorm2d: 1-9                                 [8, 16, 21, 21]           (recursive)
├─MaxPool2d: 1-10                                  [8, 16, 10, 10]           --
├─Conv2d: 1-

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchtsmixer import TSMixer

# Model parameters
sequence_length = 10
prediction_length = 5
input_channels = 2
output_channels = 1

# Create the TSMixer model
model = TSMixer(sequence_length, prediction_length, input_channels, output_channels)

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Dummy dataset (replace with real data)
# Assuming batch_size, seq_len, num_features format
X_train = torch.randn(10, 32, sequence_length, input_channels)
y_train = torch.randn(10, 32, prediction_length, output_channels)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for X, y in zip(X_train, y_train):
        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X)
        loss = criterion(outputs, y)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

print("Training complete")

Epoch [1/10], Loss: 1.0897
Epoch [2/10], Loss: 1.0801
Epoch [3/10], Loss: 1.0593
Epoch [4/10], Loss: 1.0388
Epoch [5/10], Loss: 1.0802
Epoch [6/10], Loss: 1.0438
Epoch [7/10], Loss: 1.0529
Epoch [8/10], Loss: 1.0678
Epoch [9/10], Loss: 1.0428
Epoch [10/10], Loss: 1.0567
Training complete


In [10]:
y.shape

torch.Size([32, 5, 1])